<a href="https://colab.research.google.com/github/pratik04032/pratik04032/blob/main/Deep_Fake_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download(
    "sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset"
)

print("Dataset Path:", path)

Using Colab cache for faster access to the 'deep-fake-detection-dfd-entire-original-dataset' dataset.
Dataset Path: /kaggle/input/deep-fake-detection-dfd-entire-original-dataset


In [2]:
import os
import cv2
import numpy as np
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from collections import defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2

# Performance optimization
tf.keras.mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(True)

In [3]:
BASE_PATH = path

REAL_PATH = os.path.join(BASE_PATH, "DFD_original sequences")
FAKE_PATH = os.path.join(BASE_PATH, "DFD_manipulated_sequences", "DFD_manipulated_sequences")

assert os.path.exists(REAL_PATH)
assert os.path.exists(FAKE_PATH)

print("✅ Paths verified")

✅ Paths verified


In [4]:
IMG_SIZE = 128
FRAME_COUNT = 10
BATCH_SIZE = 32
CLASS_NAMES = ["Real", "Fake"]

In [5]:
def extract_frames(video_path):

    cap = cv2.VideoCapture(video_path)
    frames = []

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(total // FRAME_COUNT, 1)

    for i in range(FRAME_COUNT):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
        frame = preprocess_input(frame)
        frames.append(frame)

    cap.release()
    return np.array(frames)

In [6]:
real_videos = sorted(os.listdir(REAL_PATH))
fake_videos = sorted(os.listdir(FAKE_PATH))

fake_videos = fake_videos[:len(real_videos)]

real_train, real_test = train_test_split(real_videos, test_size=0.3, random_state=42)
fake_train, fake_test = train_test_split(fake_videos, test_size=0.3, random_state=42)

print("Train:", len(real_train), len(fake_train))
print("Test:", len(real_test), len(fake_test))

Train: 254 254
Test: 110 110


In [7]:
def load_videos(video_list, label, base_path):

    X, y, ids = [], [], []

    for vid in tqdm(video_list):
        frames = extract_frames(os.path.join(base_path, vid))
        if len(frames) != FRAME_COUNT:
            continue

        for f in frames:
            X.append(f)
            y.append(label)
            ids.append(vid)

    return np.array(X), np.array(y), np.array(ids)

In [ ]:
X_real, y_real, _ = load_videos(real_train, 0, REAL_PATH)
X_fake, y_fake, _ = load_videos(fake_train, 1, FAKE_PATH)

X_train = np.concatenate([X_real, X_fake])
y_train = np.concatenate([y_real, y_fake])

  3%|▎         | 7/254 [01:23<51:31, 12.52s/it]